In [2]:
import json
import requests
import pandas as pd
from pydantic import BaseModel

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3:0.6b"

def ask_ollama(messages, format_schema=None):
    payload = {
        "model":MODEL,
        "messages":messages,
        "stream":False
    }

    if format_schema:
        payload["format"] = format_schema

    response = requests.post(
        OLLAMA_URL,
        json=payload
    )

    return response.json()["message"]["content"]

#Task 1 Sentiment Classification
review = """
The hotel room was spacious and clean.
Staff were friendly.
Breakfast was average though.
Overall I would definitely stay again.
"""

messages = [
    {
        "role":"user",
        "content":f"""
Classify the sentiment of this review.

{review}
"""
    }
]
result = ask_ollama(messages)

print(result)

The sentiment of the review is **positive**. The review highlights positive aspects (spacious and clean rooms, friendly staff, and a positive overall recommendation to stay again) while acknowledging a neutral statement about breakfast.


In [4]:
messages = [
{
"role":"user",
"content":"""
Example 1

Text:
I hate this phone.

Sentiment:
Negative

Example 2

Text:
Amazing service and great staff.

Sentiment:
Positive

Now classify:

The hotel room was spacious and clean.
Staff were friendly.
Breakfast was average though.
Overall I would definitely stay again.
"""
}
]

print(ask_ollama(messages))

Sentiment: Positive  

The text describes positive aspects of the hotel experience: spacious and clean rooms, friendly staff, average breakfast, and a strong intention to return. All elements contribute positively to the overall sentiment.


In [5]:
messages = [
{
"role":"user",
"content":f"""
Read the review carefully.

Step 1:
Identify positive statements.

Step 2:
Identify negative statements.

Step 3:
Compare them.

Step 4:
Give final sentiment.

Review:

{review}
"""
}
]

print(ask_ollama(messages))

**Step 1:**  
Identify positive statements:  
- "spacious and clean"  
- "staff were friendly"  
- "average though" (though this is a positive note)  
- "overall I would definitely stay again"  

**Step 2:**  
Identify negative statements:  
- None (no explicit negative words).  

**Step 3:**  
Compare positive and negative statements.  

**Step 4:**  
**Final sentiment:** Positive.


In [6]:
#Task 2 Entity Extraction
text = """
John Smith works at Microsoft in Seattle.
He met Sundar Pichai on Monday.
"""

messages = [
{
"role":"user",
"content":f"""
Extract all entities.

{text}
"""
}
]
print(ask_ollama(messages))

- John Smith  
- Microsoft  
- Seattle  
- Sundar Pichai


In [3]:
messages = [
{
"role":"user",
"content":"""
Example

Sentence:
Alice works at Google in London.

Entities

Person:
Alice

Organization:
Google

Location:
London

Now do the same:

John Smith works at Microsoft in Seattle.
He met Sundar Pichai on Monday.
"""
}
]

print(ask_ollama(messages))

Person: John Smith  
Organization: Microsoft  
Location: Seattle  

The entities extracted from the sentences are:  
- **Person**: John Smith  
- **Organization**: Microsoft  
- **Location**: Seattle  

The additional mention of Sundar Pichai is an event or action, not an entity.


In [4]:
messages = [
{
"role":"user",
"content":"""
Identify entities step by step.

Step 1:
Find all persons.

Step 2:
Find organizations.

Step 3:
Find locations.

Step 4:
Return grouped entities.

John Smith works at Microsoft in Seattle.
He met Sundar Pichai on Monday.
"""
}
]

print(ask_ollama(messages))

1. **Persons**: John Smith  
2. **Organizations**: Microsoft  
3. **Locations**: Seattle  

**Grouped Entities**:  
- **Person**: John Smith  
- **Organization**: Microsoft  
- **Location**: Seattle


In [10]:
#Task 3
article = """
Artificial Intelligence is rapidly transforming healthcare.
Hospitals now use AI for diagnosis,
medical imaging,
drug discovery,
patient monitoring,
and administrative automation.

Researchers believe AI will improve healthcare quality,
reduce costs,
and help doctors make faster decisions.
"""

messages = [
{
"role":"user",
"content":f"""
Summarize this article.

{article}
"""
}
]

print(ask_ollama(messages))

Artificial Intelligence is transforming healthcare by enabling tasks like diagnosis, medical imaging, drug discovery, patient monitoring, and administrative automation, with researchers predicting improved quality, cost reduction, and faster decision-making.


In [11]:
messages = [
{
"role":"user",
"content":"""
Example

Text:
Python is easy to learn.
It is popular in data science.

Summary:
Python is a popular programming language widely used in data science.

Now summarize:

Artificial Intelligence is rapidly transforming healthcare...
"""
}
]

print(ask_ollama(messages))

Artificial Intelligence is rapidly transforming healthcare, leading to advancements in medical treatments and diagnostics.


In [12]:
messages = [
{
"role":"user",
"content":"""
Read carefully.

Step 1:
Identify main topic.

Step 2:
Identify supporting ideas.

Step 3:
Write a concise summary.

Artificial Intelligence is rapidly transforming healthcare...
"""
}
]

print(ask_ollama(messages))

**Main Topic:** Artificial Intelligence is rapidly transforming healthcare.  

**Supporting Ideas:**  
1. AI improves medical diagnosis and treatment by analyzing patient data.  
2. It streamlines patient monitoring and healthcare delivery.  
3. Ethical concerns around privacy and data security are being addressed.  
4. Future applications include personalized medicine and predictive health analytics.  

**Summary:**  
Artificial intelligence is revolutionizing healthcare through advancements in diagnostics, treatment planning, and patient monitoring. It enhances efficiency and accuracy while raising ethical and privacy-related challenges. As AI evolves, its role in personalized medicine and predictive health strategies promises to further optimize patient care.


In [6]:
messages = [
{
"role":"user",
"content":"""
Tell me everything about this review.

The hotel was nice.
"""
}
]

print(ask_ollama(messages))

The review highlights that the hotel was a pleasant experience. Here's a breakdown:  

- **Positive Aspects**: The user mentions the hotel’s cleanliness, service, or amenities. For example, if the hotel has well-maintained facilities or friendly staff, that's a strong point.  
- **Personal Insight**: The reviewer might have included specific details, like a welcoming lobby or exceptional amenities, which contribute to the overall satisfaction.  
- **Conclusion**: The review emphasizes the positive experience, suggesting that the hotel was well-liked and worth a stay.  

In summary, the review is straightforward, focusing on the hotel’s positive attributes without additional details.


In [15]:
class Sentiment(BaseModel):
    sentiment:str
    confidence:float

schema = Sentiment.model_json_schema()

messages = [
{
    "role":"user",
    "content":"""
    Content sentiment
    Review:

    The hotel was nice.
    Return JSON only
    """
}
]

response = ask_ollama(messages, schema)
print(response)
parsed = Sentiment.model_validate_json(response)
print(parsed)



{"sentiment": "positive", "confidence": 1}
sentiment='positive' confidence=1.0


In [17]:
comparison = pd.DataFrame({
    "Technique":[
        "Zero Shot",
        "Few Shot",
        "Chain of Thought"
    ],
    "Strength":[
        "Simple and fast",
        "Better consistency",
        "Best reasoning and explanation"
    ],
    "Weakness":[
        "May be incosistent",
        "Longer prompts",
        "Higher token usage"
    ]
})

comparison

,Technique,Strength,Weakness
0,Zero Shot,Simple and fast,May be incosistent
1,Few Shot,Better consistency,Longer prompts
2,Chain of Thought,Best reasoning and explanation,Higher token usage
